# Task 2: Knowledge distillation
Teacher (MLP) vs hard-label student vs distilled student, loss `alpha*CE + (1-alpha)*T^2*KL(teacher||student)`. Sweep T and alpha; select on **dev**; report test, size, latency.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config
from src.data import load_base, make_split
from src.features import attach_views

DATA_SOURCE = "local"   # "local" (data/raw TSVs), "hf", or "synthetic" (smoke test only)
cfg = load_config(overrides={"data.source": DATA_SOURCE})
base = attach_views(load_base(cfg), cfg)
cfg["_n_classes"] = base.n_classes
print(base.label_names, {k: len(v) for k, v in base.texts.items()})

In [ ]:
from task2_distillation.distill import run_distillation, select_best
cfg['task2']['seeds'] = [0, 1, 2]
df = run_distillation(base, cfg, cfg['task2']['seeds'])
best = select_best(df); print('best on dev:', best)

In [ ]:
d = df[df.model == 'student_distilled'].groupby(['T', 'alpha'])[['dev_macro_f1', 'test_macro_f1']].mean().round(4); d

In [ ]:
chosen = df[(df.model == 'student_distilled') & (df['T'] == best['T']) & (df.alpha == best['alpha'])].assign(model='student_distilled(best_on_dev)')
comp = pd.concat([df[df.model.isin(['teacher', 'student_hard', 'baseline_charLR'])], chosen])
table = comp.groupby('model')[['test_macro_f1', 'test_ece', 'params', 'size_kb', 'latency_ms']].agg(['mean', 'std']).round(4); table

In [ ]:
m = comp.groupby('model')['test_macro_f1'].mean()
plt.figure(figsize=(6, 3.5)); m.plot(kind='bar'); plt.ylabel('test macro-F1'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

Full run that also writes `results/task2/`: `python -m task2_distillation.run`.